<a href="https://colab.research.google.com/github/RobinSmits/Schaapje/blob/main/Schaapje_2B_Pretrained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install and Import Modules

In [1]:
# Install Modules
!pip install -q accelerate==1.2.1
!pip install -q bitsandbytes==0.45.0
!pip install -q datasets==3.1.0
!pip install -q peft==0.14.0
!pip install -q transformers==4.47.0
!pip install -q trl==0.12.2
!pip install -q flash-attn==2.7.2.post1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 19.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 61.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.3 MB/s eta 0:00:00
   ━━━

In [2]:
# Import Modules
from datasets import load_dataset, load_from_disk, interleave_datasets
from huggingface_hub import notebook_login
from transformers import (AutoTokenizer,
                          AutoModelForCausalLM,
                          DataCollatorForLanguageModeling,
                          TrainingArguments)
import torch
from trl import SFTTrainer, SFTConfig

# Set TF32 for A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

## Constants

In [3]:
# Set Name Constants
model_name = 'ibm-granite/granite-3.0-2b-instruct'
hf_model_name = 'Schaapje-2B-Pretrained'

## Connect Google Drive

In [4]:
# Mount Google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

# Set Folder to use...
WORK_DIR = '/content/drive/My Drive/Schaapje/'
os.makedirs(WORK_DIR, exist_ok = True)

Mounted at /content/drive


## HuggingFace Login

In [5]:
# HuggingFace Hub Login
notebook_login()

## Tokenizer

In [6]:
# Create Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Max Length
MAX_LEN = 4096

# Tokenizer Summary
print(tokenizer)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/5.64k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

GPT2TokenizerFast(name_or_path='ibm-granite/granite-3.0-2b-instruct', vocab_size=49152, model_max_length=9223372036854775807, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>', 'additional_special_tokens': ['<|start_of_role|>', '<|end_of_role|>', '<|tool_call|>']}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<fim_prefix>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<fim_middle>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<fim_suffix>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<fim_pad>", rstrip=False, lstrip=False, single_word=False,

## Create Model based on IBM Granite 3.0 2B Instruct Model

In [7]:
# Create Model
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map = "auto",
                                             attn_implementation = "flash_attention_2",
                                             torch_dtype = torch.bfloat16)

# Set cache to False
model.config.use_cache = False

# Show Model Summary
print(model)

config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/29.9k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

GraniteForCausalLM(
  (model): GraniteModel(
    (embed_tokens): Embedding(49155, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-39): 40 x GraniteDecoderLayer(
        (self_attn): GraniteFlashAttention2(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GraniteMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): GraniteRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): GraniteRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): GraniteRMSNorm((

## Datasets

In [8]:
# Load Train Dataset
train_data = load_dataset("robinsmits/pretrain_dataset_v1", split = 'train_2') # 5 splits available: train_1, train_2, train_3, train_4, train_5

# Summary
print(train_data)

README.md:   0%|          | 0.00/891 [00:00<?, ?B/s]

train_1-00000-of-00004.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

train_1-00001-of-00004.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

train_1-00002-of-00004.parquet:   0%|          | 0.00/282M [00:00<?, ?B/s]

train_1-00003-of-00004.parquet:   0%|          | 0.00/235M [00:00<?, ?B/s]

...

Generating train_3 split:   0%|          | 0/854390 [00:00<?, ? examples/s]

Generating train_4 split:   0%|          | 0/854390 [00:00<?, ? examples/s]

Generating train_5 split:   0%|          | 0/854394 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/16189 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 854390
})


In [9]:
# Load Train Dataset
test_data = load_dataset("robinsmits/pretrain_dataset_v1", split = 'test')

# Summary
print(test_data)

Dataset({
    features: ['text'],
    num_rows: 16189
})


## Samples

In [10]:
print(train_data[0]['text'][:128])
print(test_data[0]['text'][:128])

Straat Roti (Indonesisch: Selat Roti of Selat Rote) is een zeestraat in de Indonesische provincie Oost-Nusa Tenggara. De zeestra
Joden in Amsterdam die tijdens de Tweede Wereldoorlog verplicht in een getto moesten wonen, kregen een huurverhoging opgelegd va


## Train Model

In [ ]:
# Set Steps
eval_steps = 100
save_steps = 100
logging_steps = 50

# Set TrainingArguments
training_args = SFTConfig(max_steps = 2400, # Roughly 1 epoch with all 5 data splits combined
                          eval_strategy = 'steps',
                          logging_steps = logging_steps,
                          save_strategy = 'steps',
                          eval_steps = eval_steps,
                          save_steps = save_steps,
                          save_total_limit = 6,
                          per_device_train_batch_size = 4,
                          per_device_eval_batch_size = 4,
                          gradient_accumulation_steps = 64,
                          gradient_checkpointing = True,
                          gradient_checkpointing_kwargs = {'use_reentrant': False},
                          learning_rate = 5.0e-6,
                          lr_scheduler_type = 'polynomial',
                          lr_scheduler_kwargs = {'power': 1, 'lr_end': 1.0e-6},
                          warmup_steps = 200,
                          weight_decay = 0.01,
                          max_grad_norm = 1.0,
                          optim = 'adamw_bnb_8bit',
                          bf16 = True,
                          tf32 = True,
                          dataset_text_field = 'text',
                          max_seq_length = MAX_LEN,
                          eval_packing = True,
                          packing = True,
                          dataset_num_proc = 8,
                          output_dir = f'{WORK_DIR}{hf_model_name}',
                          hub_model_id = hf_model_name,
                          push_to_hub = True,
                          hub_private_repo = True,
                          report_to = 'tensorboard')

# Config SFTTrainer
trainer = SFTTrainer(model,
                     train_dataset = train_data,
                     eval_dataset = test_data,
                     tokenizer = tokenizer,
                     data_collator = DataCollatorForLanguageModeling(tokenizer, mlm = False),
                     args = training_args)

# Perform Training
trainer.train(resume_from_checkpoint = True)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:403: UserWarning: You passed a processing_class with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `processing_class.padding_side = 'right'` to your code.
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:428: UserWarning: You passed `packing=True` to the SFTTrainer/SFTConfig, and you are training your model with `max_steps` strategy. The dataset will be iterated until the `max_steps` are reached.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
/usr/local/lib/python3.10/dist-packages/transformers/trainer.py:3354: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the defaul

Step,Training Loss,Validation Loss
700,1.938100,2.021713
800,1.926800,2.013805
900,1.917700,2.008336
1000,1.913500,2.003886


Step,Training Loss,Validation Loss
700,1.938100,2.021713
800,1.926800,2.013805
900,1.917700,2.008336
1000,1.913500,2.003886
1100,1.907200,2.000670
